# Inspect one session (neuro+behavior) 

#### Get the recording of one session

In [ ]:
%load_ext autoreload
%autoreload 2

from src.io import load_binary_session
import spikeinterface.preprocessing as spr
import probeinterface as pi

subject = "Router"
session = "Router_20211130"
recording = load_binary_session(subject, session)
sig = recording.get_traces(start_frame=0, end_frame=1000)
# Load the probe configuration from the JSON file and attach it to the recording object
probegroup = pi.read_probeinterface('../utils/probe_layout.json')
recording = recording.set_probegroup(probegroup)
recording_uV = spr.scale(recording, gain=1e6)
duration_s = recording.get_duration()
print(f"Num channels: {recording.get_num_channels()} - Duration: {duration_s:.1f} s = {duration_s / 60:.1f} min - Sampling frequency: {recording.get_sampling_frequency()} Hz")

#### Visually inspect raw data

In [ ]:
%matplotlib widget
import spikeinterface.widgets as sw

# Load the raw recording
sw.plot_traces(recording_uV, time_range=(0, 5), mode='map', backend='ipywidgets', clim=(-20, 20))

#### Visualize the mean spectrum of that session

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

# Extract 5 seconds of data to achieve a clean 1 Hz frequency resolution
fs = recording_uV.get_sampling_frequency()
end_frame_5s = int(5 * fs)
traces = recording_uV.get_traces(start_frame=0, end_frame=end_frame_5s)

# Compute Power Spectral Density using Welch's method 
# nperseg=int(fs) forces the window size to exactly 1 second (1 Hz resolution)
frequencies, psd = welch(traces, fs=fs, nperseg=int(fs), axis=0)

# Average the PSD across all 128 channels to emphasize global common-mode noise
mean_psd = np.mean(psd, axis=1)

# Plot the spectrum focusing on the [0, 150] Hz range
plt.figure(figsize=(10, 5))
plt.semilogy(frequencies, mean_psd, color='black', linewidth=1.2)

# Highlight fundamental power-line frequency and its first harmonic
plt.axvline(x=50, color='red', linestyle='--', alpha=0.7, label='50 Hz')
plt.axvline(x=100, color='orange', linestyle=':', alpha=0.7, label='100 Hz')

plt.xlabel('Frequency (Hz)')
plt.ylabel(r'Power Spectral Density ($\mu V^2$/Hz)')
plt.title('Averaged PSD of Raw Signal')
plt.legend(frameon=False)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Inspect behavior

In [1]:
from src.io import inspect_behavior

# Event counts per action class and hand, for every session in session_metadata.csv
inspect_behavior()

[Router | Router_20211130] 27 step(s) without WalkNumber: counted in 'steps', never in 'first_step'.


Hemisphere grasp_hook      grasp_floor       \
                                       contra ipsi      contra ipsi   
Subject Session                                                       
Router  Router_20211130          L         19    7           5   16   
        Router_20220211          L          5   13          10    4   
        Total                              24   20          15   20   
Wifi    Wifi_20210618            L          4   12          12   56   
        Wifi_20221020            L          1    1          11   12   
        Total                               5   13          23   68   
All     Total                              29   33          38   88   

                        first_step       steps       
                            contra ipsi contra ipsi  
Subject Session                                      
Router  Router_20211130         15    6     73   72  
        Router_20220211         17    9    103   97  
        Total                   32   15    176  169  
Wifi    Wifi_20210618            2    5     24   28  
        Wifi_20221020            6    7     92   92  
        Total                    8   12    116  120  
All     Total                   40   27    292  289